# 02 近傍法 — 「似た人」「似た商品」から薦める

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約30秒

## ゴール
- **近傍法**（協調フィルタリング）の仕組みを理解し、自分の手で作って動かす。
- 近傍法に**何を入力するか**（評価した／好き）で、**何を学ぶか**が変わることを確かめる。
- 近傍法が、**人気順・平均評価順**と比べて、どうなのかを、**観測テストと本当の好みの両方**で測る。
- 「個人化」は、データが少ないと**効かない**ことを知る。

## 前のNotebookとのつながり
00・01 で、基準（人気順・平均評価順）と、評価の落とし穴を学びました。
ここでは、最初の「個人化」の手法を作ります。

## 近傍法とは
**協調フィルタリング**は、**みんなの行動**から、おすすめを作ります。商品の中身（ジャンルなど）は使いません。

- **アイテム型**: 商品 i と j が、**同じユーザーに一緒に評価される**ほど「似ている」。あなたが j を好きなら、**j に似た i** を薦める。
- **ユーザー型**: あなたに**似たユーザー**（同じ商品を評価した人）を探し、**その人が好きな商品**を薦める。

「似ている」は、**コサイン類似度**で測ります（共起の数を、それぞれの評価数で割ったもの）。

## キーワード
- **共起**: 同じユーザーに、両方の商品が評価された数。
- **コサイン類似度**: 共起 ÷ √(商品iの評価数 × 商品jの評価数)。評価数の多い商品の影響を、割って弱める。
- **α**: 評価数での割り方（0.5 が通常のコサイン）。**収縮（shrink）**: 共起が少ない類似度を、小さくする（偶然の類似を、信用しない）。
- **近傍数 k**: 似ている上位k個だけを使う。
- **入力**: **評価した**（何を評価したか）／**好き**（評価が4以上）。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from recsys_synth import make_dataset
from recsys_eval import exclusion_mask, topk_items, evaluate_observed, evaluate_true, popularity_scores
from recsys_models import interaction_matrix, item_similarity, item_knn_scores
import run_phase2

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

d = make_dataset(1_000_000)
w, tr = d.world, d.train()
U, I = w.n_users, w.n_items
B = interaction_matrix(tr, U, I, "interactions")
sim = item_similarity(B, alpha=0.5, shrink=5.0)

# ある商品（評価された回数が真ん中あたりの商品）に、似ている商品を見る
cnt = B.sum(axis=0)
i = int(np.argsort(cnt)[len(cnt) // 2])
top = np.argsort(-sim[i])[:5]
rows = [{"商品": j, "類似度": round(float(sim[i, j]), 3), "ジャンル": int(w.item_genre[j]), "評価された回数": int(cnt[j])} for j in top]
print(f"商品 {i}（ジャンル {int(w.item_genre[i])}・評価された回数 {int(cnt[i])}）に似た商品の上位5つ:")
pd.DataFrame(rows).set_index("商品")

**何をしているか**: アイテム型の**類似度**を作り、ある商品に似た商品の上位5つを見ます。

**出力の見方**: 似た商品の上位5つには、**評価された回数が多い商品**（457回・370回）が2つ入る一方、**少ない商品**（12〜30回）も混ざります。**共起**（同じユーザーに一緒に評価された回数）は、**人気の商品どうしほど大きくなりやすい**ので、人気の商品が似た商品に挙がりやすい傾向があります。
また、この商品は評価が**19回**と少なく、似た商品のジャンルも**ばらばら**（この商品と同じジャンルは1つだけ）です。**評価の少ない商品の類似度は、あてになりません**。**収縮**は、この偶然の類似を、小さくするための工夫です。

In [ ]:
e1 = run_phase2.e1()
e1.set_index("手法")

## E1. 近傍法を、基準と比べる（世界10回の平均±標準誤差）

**何をしているか**: アイテム型・ユーザー型の近傍法を、**2通りの入力**（評価した／好き）で作り、基準と比べます。設定は、通常のコサイン（α=0.5）・収縮5・近傍は全部、で固定です。

**出力の見方**:
- **「評価した」を入力にした近傍法は、人気順とほぼ同じ**: 観測NDCG 0.166〜0.167（人気順 0.168）、本当のprecision 0.31〜0.32（人気順 0.31）。**何を評価したか**（露出）を学ぶと、**人気のものを薦める**ことになります。「個人化」したつもりが、**ほとんど人気順**です。
- **「好き」（評価4以上）を入力にすると、本当の好みでの当たりが上がります**: アイテム型 **0.495**、ユーザー型 **0.472**（人気順 0.305）。カバレッジも **0.71／0.61**（人気順 0.07）と、**幅広く**薦めます。観測NDCGは、0.180（アイテム型）〜0.186（ユーザー型）と、人気順（0.168）と同程度か、わずかに上（標準誤差0.008で、差は約1.5倍〜2倍）。
- **ところが、平均評価順（0.616）に、及びません**。**個人化したのに、個人化しない「平均評価順」より、本当の好みの当たりが低い**のです。理由は、次のE3で調べます。
- **上限**（0.973）との差は、まだ大きい。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
names = list(e1["手法"])
obs = [float(x.split(" ")[0]) for x in e1["観測NDCG@10"]]
tru = [float(x.split(" ")[0]) for x in e1["本当のprecision@10"]]
x = np.arange(len(names))
ax.bar(x - 0.2, np.array(obs) / max(obs), 0.4, label="観測NDCG（最大を1にそろえる）")
ax.bar(x + 0.2, tru, 0.4, label="本当のprecision")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
ax.set(title="観測テストと、本当の好みで、評価が違う")
fig.tight_layout()

**何をしているか**: E1の結果を、観測テスト（観測NDCG。最大を1にそろえた）と、本当の好み（precision）で、並べて描きます。

**出力の見方**: **人気順・近傍法（評価した）**は、**観測の棒は高い**のに、**本当の好みの棒は低い**。**平均評価順・上限**は、その逆（観測は低く、本当は高い）。**近傍法（好き）**は、その中間です。**観測テストだけで評価すると、人気に偏った手法を、良いと誤解します**。

In [ ]:
grid, sel = run_phase2.e2()
sel.set_index(["型", "入力", "選び方"])

## E2. ハイパーパラメータは、何で選ぶか

**何をしているか**: α（5通り）・収縮（3通り）・近傍数k（4通り）の**60通り**を、1つの世界（開発用）で試し、各（型×入力）について、
- **観測テストで最良**の設定
- **本当の好みで最良**の設定

を選びます。それぞれの設定を、**別の9つの世界**で評価して、結果を比べます。

**出力の見方**:
- **観測テストで選ぶと、本当の好みの当たりが、下がります**。アイテム型（好き）で、観測で選ぶと本当のprecision **0.471**、本当の好みで選ぶと **0.500**（約0.03の差）。ユーザー型（好き）で **0.459 対 0.491**。
- 逆に、**観測テストの数字は、観測で選んだほうが高い**（0.181 対 0.164）。つまり、**何で選ぶかが、結果を決めます**。
- 観測で選ばれる設定は、**人気を強く効かせる設定**（α=1 など）に偏ります。**観測データが人気に偏っている**ためです。
- 現実には、**本当の好みは分かりません**。**観測テストで選ぶ**しかないので、このずれは**避けられない**。だから、**選んだ設定が、観測の偏りに引きずられていないか**、他の指標（カバレッジ・多様性）も合わせて、確認します。
- 「評価した」を入力にした近傍法では、どの設定でも、**本当のprecisionは0.31〜0.32のまま**（ほぼ変わらない）。**設定の工夫では、「何を入力するか」の差は、埋まりません**。

In [ ]:
g = grid[(grid["型"] == "アイテム型") & (grid["入力"] == "liked") & (grid["shrink"] == 5.0) & (grid["k"].isna())].sort_values("alpha")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(g["alpha"], g["観測"], "o-"); ax[0].set(title="アイテム型（好き）: α と 観測NDCG", xlabel="α（大きいほど、人気の商品の影響が強い）"); ax[0].grid(alpha=0.3)
ax[1].plot(g["alpha"], g["真"], "o-", color="C1"); ax[1].set(title="α と 本当のprecision", xlabel="α"); ax[1].grid(alpha=0.3)
fig.tight_layout()
g[["alpha", "観測", "真"]].round(4).set_index("alpha")

**何をしているか**: アイテム型（好き）で、**α**だけを変えたときの、観測NDCGと本当のprecisionを見ます（収縮5・近傍は全部）。

**出力の見方**:
- **観測NDCG**は、αが大きいほど**上がり続けます**（0.037 → 0.214）。αが大きいほど、人気の商品が強く効く設定で、**観測が人気に偏っている**ので、観測では良く見えます。
- **本当のprecision**は、**α=0.5付近が最良**（0.514）で、αが大きすぎても、小さすぎても下がります。α=0（人気の商品を、極端に抑える）は、**観測も本当も悪い**（0.037／0.406）。
- **αは、人気にどれだけ寄せるかの「つまみ」**。観測テストだけを見ると、**つまみを人気側に回しすぎる**ことになります。

In [ ]:
e3 = run_phase2.e3()
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(e3))
for name, c in [("人気順", "C7"), ("平均評価順", "C2"), ("アイテム型・好き(4以上)", "C0"), ("ユーザー型・好き(4以上)", "C1")]:
    ax.plot(x, e3[name], "o-", label=name, color=c)
ax.set_xticks(x); ax.set_xticklabels(e3["学習期間の評価数"]); ax.set(xlabel="学習期間のユーザーの評価数", ylabel="本当のprecision@10", title="評価が多いユーザーほど、近傍法は良くなる？")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
e3.set_index("学習期間の評価数")

## E3. 個人化は、データが多いほど効くのか

**何をしているか**: ユーザーを、学習期間の**評価数**で5つの帯に分けて、本当のprecisionを比べます（世界10回の平均）。

**出力の見方**:
- **平均評価順は、すべての帯で、近傍法（好き）を上回ります**（0.57〜0.63 対 0.42〜0.52）。**評価が多い（40件以上）ユーザーでも、同じ**です。
- 近傍法は、**評価が多いほど良くなる、とは限りません**。評価数5〜9件で最良（アイテム型 **0.518**）で、40件以上では **0.469** に下がります。評価が多いユーザーは、**いろいろな商品を評価している**ので、「似た人」「似た商品」の信号が、**ぼやけます**。また（推測）、**良い商品を、すでに消費している**ので、残りの候補に、好きなものが減ります。
- **評価が0〜4件のユーザー（新しいユーザー）**でも、近傍法（好き）は0.42〜0.43で、人気順（0.31）より高い。ただし、**学習期間に評価が0件のユーザー**（約6人）には、**スコアが出ません**（**コールドスタート**。Phase4）。
- 結論: 近傍法の「個人化」は、**この疎なデータ（密度5%）では、商品の質（平均評価）を、上回れません**。**個人化が効くには、データが足りない**、あるいは、**手法が、評価の大きさ（点数）を、十分に使えていない**。次の Phase3（行列分解）で、評価の**点数**を使います。

## 試してみよう（演習）
1. **入力を変える**: E1の設定を、`mode="liked"` の閾値（評価4以上）を、`neighbors.py` の `interaction_matrix` で「5だけ」に変えて（`d["rating"] >= 5`）、結果がどう変わるか見てください。分かるのは、「好き」の定義が、結果を決めることです。
2. **収縮を変える**: `DEFAULT` の `shrink=5.0` を `0` と `50` にして、E1を実行してください（`run_phase2.DEFAULT["shrink"] = 50` のように、セルで代入）。見るポイントは、カバレッジと、本当のprecisionです。分かるのは、収縮が大きいほど、共起の少ない商品が、薦められにくくなることです。
3. **近傍数kを小さく**: `DEFAULT["k"] = 5` にして、E1を実行してください。分かるのは、近傍を絞ると、類似度の信号が、鋭くなる反面、ノイズも増えることです。
4. **別の世界で試す**: `make_dataset(1_000_000)` を `1_000_001` に変えて、アイテム型の類似度の上位が、どう変わるか見てください。

## まとめ
- **近傍法**は、「似た人・似た商品」から薦める。**入力が、学ぶものを決める**: **「評価した」を入力にすると、人気順とほぼ同じ**（観測の露出を学ぶ）。**「好き（4以上）」を入力にすると**、本当の好みの当たりが上がる（0.31→0.50）。
- **ハイパーパラメータを、観測テストで選ぶと**、人気側に寄りすぎて、**本当の好みの当たりが約0.03下がる**。観測テストの数字だけで、設定を選ばない。
- **近傍法は、平均評価順（個人化なし）に、及ばなかった**（0.50 対 0.62）。疎なデータでは、**個人化は、簡単には効かない**。評価の数が増えても、近傍法は良くならない。
- **新しいユーザー**（評価0件）には、使えない（コールドスタート）。

次の Phase3 では、**行列分解**（ユーザーと商品を、少数の「潜在因子」で表す）で、評価の**点数**を使い、個人化が効くかを確かめます。